Imports and Config

In [1]:
import pandas as pd
import wbgapi as wb
import requests
from io import BytesIO
from pathlib import Path

# Paths — using pathlib so it works on any OS
RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


Pulling Kenya's World Bank indicators

In [15]:
import requests
import zipfile
from io import BytesIO
from pathlib import Path

RAW = Path("../data/raw/worldbank")
RAW.mkdir(parents=True, exist_ok=True)

INDICATORS = [
    "SH.H2O.SMDW.ZS",
    "SH.H2O.SMDW.UR.ZS",
    "SH.H2O.SMDW.RU.ZS",
    "SH.STA.BASS.ZS",
    "SH.STA.WASH.P5",
    "ER.H2O.FWST.ZS",
]

def download_indicator_csv(indicator_code):
    url = (
        f"https://api.worldbank.org/v2/en/indicator/{indicator_code}"
        f"?downloadformat=csv"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    # Response is a zip containing 3 CSVs
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        # The data file is the one NOT starting with Metadata
        data_file = [
            f for f in z.namelist()
            if f.endswith(".csv") and not f.startswith("Metadata")
        ][0]
        z.extract(data_file, RAW)
        print(f"Saved: {data_file}")

for ind in INDICATORS:
    try:
        download_indicator_csv(ind)
    except Exception as e:
        print(f"Failed {ind}: {e}")

Saved: API_SH.H2O.SMDW.ZS_DS2_en_csv_v2_630.csv
Saved: API_SH.H2O.SMDW.UR.ZS_DS2_en_csv_v2_364.csv
Saved: API_SH.H2O.SMDW.RU.ZS_DS2_en_csv_v2_955.csv
Saved: API_SH.STA.BASS.ZS_DS2_en_csv_v2_213350.csv
Saved: API_SH.STA.WASH.P5_DS2_en_csv_v2_207246.csv
Saved: API_ER.H2O.FWST.ZS_DS2_en_csv_v2_12612.csv


In [16]:
import pandas as pd
from pathlib import Path
import glob

RAW = Path("../data/raw/worldbank")

dfs = []
for filepath in RAW.glob("API_*.csv"):
    df = pd.read_csv(filepath, skiprows=4)  # WB CSVs have a 4-row header
    kenya = df[df["Country Code"] == "KEN"].copy()
    if not kenya.empty:
        dfs.append(kenya)
        print(f"Loaded: {filepath.name} — {kenya.shape}")

df_wb = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {df_wb.shape}")
df_wb.head()

Loaded: API_ER.H2O.FWST.ZS_DS2_en_csv_v2_12612.csv — (1, 71)
Loaded: API_SH.H2O.SMDW.RU.ZS_DS2_en_csv_v2_955.csv — (1, 71)
Loaded: API_SH.H2O.SMDW.UR.ZS_DS2_en_csv_v2_364.csv — (1, 71)
Loaded: API_SH.H2O.SMDW.ZS_DS2_en_csv_v2_630.csv — (1, 71)
Loaded: API_SH.STA.BASS.ZS_DS2_en_csv_v2_213350.csv — (1, 71)
Loaded: API_SH.STA.WASH.P5_DS2_en_csv_v2_207246.csv — (1, 71)

Total rows: (6, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,33.239901,33.239901,33.239901,33.239901,33.239901,33.239901,NaN,NaN,NaN,NaN
1,Kenya,KEN,People using safely managed drinking water ser...,SH.H2O.SMDW.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kenya,KEN,People using safely managed drinking water ser...,SH.H2O.SMDW.UR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Kenya,KEN,People using safely managed drinking water ser...,SH.H2O.SMDW.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Kenya,KEN,People using at least basic sanitation service...,SH.STA.BASS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,35.857250,36.578389,37.298798,38.018341,38.736878,39.454253,40.170337,40.884965,NaN,NaN


In [19]:
# Identify the metadata columns vs year columns
meta_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

# All other columns are years — but filter to only valid year columns
year_cols = [col for col in df_wb.columns if str(col).strip().isdigit()]

print(f"Year columns found: {len(year_cols)} ({year_cols[0]} to {year_cols[-1]})")

Year columns found: 66 (1960 to 2025)


In [20]:
# Melt to long format
df_long = df_wb.melt(
    id_vars=meta_cols,
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

# Clean up types
df_long["year"] = df_long["year"].astype(int)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

# Drop rows where value is NaN — no point storing empty readings
df_long = df_long.dropna(subset=["value"])

print(f"Shape after melt: {df_long.shape}")
df_long.head(10)

Shape after melt: (59, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,year,value
180,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1990,16.892003
186,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1991,16.488807
192,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1992,16.085611
198,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1993,15.682415
204,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1994,15.279219
210,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1995,14.876023
216,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1996,14.472826
222,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1997,14.069630
228,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1998,13.666434
234,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,1999,13.263238


In [21]:
df_long = df_long.rename(columns={
    "Country Name":     "country_name",
    "Country Code":     "country_code",
    "Indicator Name":   "indicator_name",
    "Indicator Code":   "indicator_code",
})

# Keep only years from 2000 onwards — earlier data is too sparse to be useful
df_long = df_long[df_long["year"] >= 2000].reset_index(drop=True)

print(f"Shape after filtering to 2000+: {df_long.shape}")
df_long.head()

Shape after filtering to 2000+: (49, 6)


,country_name,country_code,indicator_name,indicator_code,year,value
0,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,2000,12.860042
1,Kenya,KEN,People using at least basic sanitation service...,SH.STA.BASS.ZS,2000,23.647380
2,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,2001,14.948739
3,Kenya,KEN,People using at least basic sanitation service...,SH.STA.BASS.ZS,2001,24.316460
4,Kenya,KEN,Level of water stress: freshwater withdrawal a...,ER.H2O.FWST.ZS,2002,17.037436


In [22]:
# How many data points per indicator?
summary = (
    df_long.groupby(["indicator_code", "indicator_name"])
    .agg(
        years_with_data=("year", "count"),
        earliest_year=("year", "min"),
        latest_year=("year", "max"),
        latest_value=("value", "last")
    )
    .reset_index()
)
print(summary.to_string(index=False))

indicator_code                                                                                            indicator_name  years_with_data  earliest_year  latest_year  latest_value
ER.H2O.FWST.ZS            Level of water stress: freshwater withdrawal as a proportion of available freshwater resources               23           2000         2022     33.239901
SH.STA.BASS.ZS                                         People using at least basic sanitation services (% of population)               25           2000         2024     40.884965
SH.STA.WASH.P5 Mortality rate attributed to unsafe water, unsafe sanitation and lack of hygiene (per 100,000 population)                1           2019         2019     29.000000


In [23]:
from pathlib import Path

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED / "wb_kenya_long.csv"
df_long.to_csv(output_path, index=False)

print(f"Saved {len(df_long)} rows to {output_path}")

Saved 49 rows to ..\data\processed\wb_kenya_long.csv


In [25]:
import requests
from pathlib import Path

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

response = requests.get(
    "https://washdata.org/data/country/KEN/household/download",
    timeout=30
)
response.raise_for_status()

raw_path = RAW / "jmp_kenya_household.xlsx"
with open(raw_path, "wb") as f:
    f.write(response.content)

print(f"Saved — {len(response.content):,} bytes")

Saved — 6,820,304 bytes


In [28]:
!pip install openpyxl


   ---------------------------------------- 0/2 [et-xmlfile]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [open

In [29]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
jmp_path = RAW / "jmp_kenya_household.xlsx"

# See all sheets first
xl = pd.ExcelFile(jmp_path)
print("Sheets:", xl.sheet_names)

Sheets: ['Key', 'Regressions', 'Introduction', 'Ladders', 'Charts SM', 'Charts B', 'Estimates', 'Chart Data', 'Data Summary', 'Water Data', 'Sanitation Data', 'Wastewater Data', 'Hygiene Data', 'Menstrual Health Data', 'Population', 'Guidance']


In [30]:
# Usually the first or second sheet — adjust name based on what prints above
df_jmp = xl.parse(xl.sheet_names[0])
print(f"Shape: {df_jmp.shape}")
print(f"Columns: {df_jmp.columns.tolist()}")
df_jmp.head(10)

Shape: (430, 69)
Columns: ['English', 'Spanish', 'French', 'Russian', 'Arabic', 'Chinese', 'Other', 'Unnamed: 7', 'Language', 'Unnamed: 9', 'Unnamed: 10', 'ISO3', 'name', 'name_FR', 'name_ES', 'name_RU', 'name_AR', 'name_CH', 'name_other', 'ISO3.1', 'Language.1', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed: 63', 'Unnamed: 64', 'Unnamed: 65', 'Unnamed: 66', 'Unnamed: 67', 'Unnamed: 68']


,English,Spanish,French,Russian,Arabic,Chinese,Other,Unnamed: 7,Language,Unnamed: 9,...,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67,Unnamed: 68
0,"Joint Monitoring Programme for Water Supply, S...",Programa Conjunto OMS/UNICEF de Monitoreo del ...,Programme commun OMS/UNICEF de suivi de l’appr...,Совместная программа по мониторингу водоснабже...,برنامج الرصد المشترك لإمدادات المياه وخدمات ال...,NaN,NaN,NaN,English,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Estimates on the use of water, sanitation and ...","Estimaciones sobre el uso de agua, saneamiento...","Estimations sur l'utilisation de l'eau, de l'a...","Оценки использования воды, санитарии и гигиены...",تقديرات حول استخدام المياه والصرف الصحي والنظا...,NaN,NaN,NaN,Spanish,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kenya,Kenya,Kenya,Кения,كينيا,0.0,0.0,NaN,French,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Updated August 2025,Actualizado en agosto de 2025,Mis à jour en août 2025,Обновлено в августе 2025 г.,تم التحديث في أغسطس 2025,NaN,NaN,NaN,Russian,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Follow the links below to find the following i...,\nSiga los enlaces a continuación para encontr...,\nSuivez les liens ci-dessous pour trouver les...,Для нахождения следующей информации используйт...,اتبع الروابط أدناه للعثور على المعلومات التالية:,NaN,NaN,NaN,Arabic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,JMP Estimates:,Estimaciones de JMP:,Estimations JMP :,Расчётные оценки СПМ,تقديرات JMP:,NaN,NaN,NaN,Chinese,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"Water, sanitation and hygiene ladders","Escaleras de agua, saneamiento e higiene","\nEchelles eau, assainissement et hygiène",Иерархические лестницы услуг в области водосна...,سلالم المياه والصرف الصحي والنظافة العامة,NaN,NaN,NaN,Other,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Safely managed services,Servicios gestionados de manera segura,Services gérés en toute sécurité,Безопасно организованные услуги,خدمات مدارة بأمان,NaN,NaN,NaN,Buttons,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"Trends in basic water, sanitation and hygiene","Tendencias en agua, saneamiento e higiene básicos","Tendances en matière d'eau, d'assainissement e...","Тенденции в базовых уровнях водоснабжения, сан...",الاتجاهات الأساسية في المياه والصرف الصحي والن...,NaN,NaN,NaN,Drinking water,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Estimates (2000-2024),Estimaciones (2000-2024),Estimations (2000-2024),Расчётные оценки (2000−2024 гг.),تقديرات (2000-2024),NaN,NaN,NaN,Sanitation,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Print all sheet names
print("All sheets:")
for i, name in enumerate(xl.sheet_names):
    print(f"  {i}: {name}")

All sheets:
  0: Key
  1: Regressions
  2: Introduction
  3: Ladders
  4: Charts SM
  5: Charts B
  6: Estimates
  7: Chart Data
  8: Data Summary
  9: Water Data
  10: Sanitation Data
  11: Wastewater Data
  12: Hygiene Data
  13: Menstrual Health Data
  14: Population
  15: Guidance


In [32]:
# Scan every sheet — print shape and first column of row 0
for name in xl.sheet_names:
    df_temp = xl.parse(name, nrows=3, header=None)
    print(f"\n--- Sheet: '{name}' --- shape hint: {df_temp.shape}")
    print(df_temp.iloc[0, :6].tolist())  # first 6 cells of first row


--- Sheet: 'Key' --- shape hint: (3, 24)
['English', 'Spanish', 'French', 'Russian', 'Arabic', 'Chinese']

--- Sheet: 'Regressions' --- shape hint: (3, 17)
['Ladders', nan, nan, nan, nan, np.float64(nan)]

--- Sheet: 'Introduction' --- shape hint: (3, 7)
[np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan)]

--- Sheet: 'Ladders' --- shape hint: (3, 9)
[np.float64(nan), np.float64(nan), nan, np.float64(nan), 'Water, sanitation and hygiene ladders', nan]

--- Sheet: 'Charts SM' --- shape hint: (3, 21)
[np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan)]

--- Sheet: 'Charts B' --- shape hint: (1, 5)
[np.float64(nan), np.float64(nan), np.float64(nan), np.float64(nan), 'Basic drinking water, sanitation and hygiene services']

--- Sheet: 'Estimates' --- shape hint: (3, 41)
['Estimates', np.float64(nan), np.float64(nan), np.float64(nan), nan, np.float64(nan)]

--- Sheet: 'Chart Data' --- shape 

In [33]:
# Read with no header first so we can see the raw structure
df_est_raw = xl.parse("Estimates", header=None)
print(f"Shape: {df_est_raw.shape}")
print("\nFirst 6 rows, first 10 columns:")
print(df_est_raw.iloc[:6, :10].to_string())

Shape: (97, 41)

First 6 rows, first 10 columns:
           0     1        2                                                   3                 4                                         5                               6                 7              8                                   9
0  Estimates   NaN      NaN                                                 NaN               NaN                                       NaN                             NaN               NaN            NaN                                 NaN
1        NaN   NaN      NaN                                                 NaN  Drinking water %                                       NaN                             NaN               NaN            NaN                                 NaN
2        NaN   NaN      NaN                                                 NaN               NaN                                       NaN                             NaN               NaN            NaN  Population using impro

In [34]:
# JMP files often have multi-row headers — scan rows 0-8
for i in range(8):
    print(f"\nRow {i}:", df_est_raw.iloc[i, :10].tolist())


Row 0: ['Estimates', nan, nan, nan, nan, nan, nan, nan, nan, nan]

Row 1: [nan, nan, nan, nan, 'Drinking water %', nan, nan, nan, nan, nan]

Row 2: [nan, nan, nan, nan, nan, nan, nan, nan, nan, 'Population using improved sources:']

Row 3: ['Country', 'Year', 'Setting', 'Population (1000s)\nSource: UN Population Division', 'Improved water', 'At least basic (improved within 30 mins)', 'Limited (improved\n > 30 mins)', 'Unimproved water', 'Surface water', 'Piped']

Row 4: ['Kenya', 2000, 'Total', 30642.89, 52.1, 45.4, 6.7, 17.4, 30.5, 31.3]

Row 5: ['Kenya', 2001, 'Total', 31619.17, 53.1, 46.2, 6.8, 16.9, 30, 31.5]

Row 6: ['Kenya', 2002, 'Total', 32629.808, 54, 47.1, 6.9, 16.4, 29.5, 31.7]

Row 7: ['Kenya', 2003, 'Total', 33652.223, 55, 48, 7.1, 16, 29, 31.8]


In [35]:
# Parse with row 3 as header
df_est = xl.parse("Estimates", header=3)

print(f"Shape: {df_est.shape}")
print(f"\nAll columns:")
for i, col in enumerate(df_est.columns):
    print(f"  {i}: {col}")

Shape: (93, 41)

All columns:
  0: Country
  1: Year
  2: Setting
  3: Population (1000s)
Source: UN Population Division
  4: Improved water
  5: At least basic (improved within 30 mins)
  6: Limited (improved
 > 30 mins)
  7: Unimproved water
  8: Surface water
  9: Piped
  10: Non-piped
  11: Accessible on premises
  12: Available when needed
  13: Free from contamination
  14: Safely managed 
drinking water
  15: Improved sanitation
  16: At least basic (improved and not shared)
  17: Limited (improved and shared)
  18: Unimproved sanitation
  19: Open defecation
  20: Latrines and other
  21: Septic tanks
  22: Sewer connection
  23: Disposed of 
in situ
  24: Emptied and treated
  25: Wastewater treated
  26: Safely managed sanitation
  27: Observed
  28: Limited 
(facility lacking water or soap)
  29: Basic 
(facility with water and soap)
  30: Observed.1
  31: On premises with water
  32: Use of menstrual materials
  33:     Reusable materials
  34:     Single-use materials
  35

In [36]:
print("\nUnique settings:", df_est["Setting"].unique())
print("\nYear range:", df_est["Year"].min(), "to", df_est["Year"].max())
print("\nFirst 10 rows:")
df_est.head(10)


Unique settings: <StringArray>
['Total', 'Rural', 'Urban']
Length: 3, dtype: str

Year range: 2000 to 2030

First 10 rows:


,Country,Year,Setting,Population (1000s)\nSource: UN Population Division,Improved water,At least basic (improved within 30 mins),Limited (improved\n > 30 mins),Unimproved water,Surface water,Piped,...,On premises with water,Use of menstrual materials,Reusable materials,Single-use materials,Enough menstrual materials,Private place to change,Awareness of menstruation before first menstrual period,Ability to reduce menstruation-related pain,Comfort seeking help from a health care provider,"Participation in school, work, social activities during menstruation"
0,Kenya,2000,Total,30642.890,52.1,45.4,6.7,17.4,30.5,31.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kenya,2001,Total,31619.170,53.1,46.2,6.8,16.9,30.0,31.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kenya,2002,Total,32629.808,54.0,47.1,6.9,16.4,29.5,31.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Kenya,2003,Total,33652.223,55.0,48.0,7.1,16.0,29.0,31.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Kenya,2004,Total,34713.455,56.0,48.8,7.2,15.5,28.5,32.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Kenya,2005,Total,35796.484,56.9,49.7,7.3,15.1,28.0,32.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Kenya,2006,Total,36904.008,57.9,50.5,7.4,14.6,27.5,32.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Kenya,2007,Total,38036.349,58.9,51.4,7.5,14.1,27.0,32.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Kenya,2008,Total,39206.351,59.8,52.3,7.6,13.7,26.5,32.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Kenya,2009,Total,40408.891,60.8,53.1,7.7,13.2,26.0,32.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Print all columns with index so we can pick
for i, col in enumerate(df_est.columns):
    print(f"{i:3}: {col}")

  0: Country
  1: Year
  2: Setting
  3: Population (1000s)
Source: UN Population Division
  4: Improved water
  5: At least basic (improved within 30 mins)
  6: Limited (improved
 > 30 mins)
  7: Unimproved water
  8: Surface water
  9: Piped
 10: Non-piped
 11: Accessible on premises
 12: Available when needed
 13: Free from contamination
 14: Safely managed 
drinking water
 15: Improved sanitation
 16: At least basic (improved and not shared)
 17: Limited (improved and shared)
 18: Unimproved sanitation
 19: Open defecation
 20: Latrines and other
 21: Septic tanks
 22: Sewer connection
 23: Disposed of 
in situ
 24: Emptied and treated
 25: Wastewater treated
 26: Safely managed sanitation
 27: Observed
 28: Limited 
(facility lacking water or soap)
 29: Basic 
(facility with water and soap)
 30: Observed.1
 31: On premises with water
 32: Use of menstrual materials
 33:     Reusable materials
 34:     Single-use materials
 35: Enough menstrual materials
 36: Private place to chang

In [38]:
KEEP_COLS = [
    "Country", "Year", "Setting",
    "Population (1000s)\nSource: UN Population Division",
    "Improved water",
    "At least basic (improved within 30 mins)",
    "Limited (improved\n > 30 mins)",
    "Unimproved water",
    "Surface water",
    "Piped",
]

# Only keep columns that actually exist
KEEP_COLS = [c for c in KEEP_COLS if c in df_est.columns]
df_est = df_est[KEEP_COLS].copy()

print(f"Shape after column selection: {df_est.shape}")

Shape after column selection: (93, 10)


In [39]:
df_est = df_est.rename(columns={
    "Country":                                          "country",
    "Year":                                             "year",
    "Setting":                                          "setting",
    "Population (1000s)\nSource: UN Population Division": "population_thousands",
    "Improved water":                                   "pct_improved_water",
    "At least basic (improved within 30 mins)":         "pct_at_least_basic_water",
    "Limited (improved\n > 30 mins)":                   "pct_limited_water",
    "Unimproved water":                                 "pct_unimproved_water",
    "Surface water":                                    "pct_surface_water",
    "Piped":                                            "pct_piped_water",
})

df_est.columns.tolist()

['country',
 'year',
 'setting',
 'population_thousands',
 'pct_improved_water',
 'pct_at_least_basic_water',
 'pct_limited_water',
 'pct_unimproved_water',
 'pct_surface_water',
 'pct_piped_water']

In [40]:
# JMP files sometimes include regional aggregates below the country rows
df_est = df_est[df_est["country"] == "Kenya"].copy()

# Drop rows where year is missing or not a valid integer
df_est = df_est[pd.to_numeric(df_est["year"], errors="coerce").notna()]
df_est["year"] = df_est["year"].astype(int)

# Filter to 2000 onwards
df_est = df_est[df_est["year"] >= 2000].reset_index(drop=True)

print(f"Shape: {df_est.shape}")
print(f"Settings: {df_est['setting'].unique()}")
print(f"Years: {df_est['year'].min()} to {df_est['year'].max()}")

Shape: (93, 10)
Settings: <StringArray>
['Total', 'Rural', 'Urban']
Length: 3, dtype: str
Years: 2000 to 2030


In [41]:
def get_setting(df, setting_name):
    return (
        df[df["setting"] == setting_name]
        .drop(columns=["setting", "country"])
        .set_index("year")
    )

df_total  = get_setting(df_est, "Total")
df_rural  = get_setting(df_est, "Rural")
df_urban  = get_setting(df_est, "Urban")

print("Total rows:", len(df_total))
print("Rural rows:", len(df_rural))
print("Urban rows:", len(df_urban))

Total rows: 31
Rural rows: 31
Urban rows: 31


In [42]:
# Suffix each setting's columns
df_total = df_total.add_suffix("_total")
df_rural = df_rural.add_suffix("_rural")
df_urban = df_urban.add_suffix("_urban")

df_jmp_clean = df_total.join([df_rural, df_urban], how="outer")
df_jmp_clean.index.name = "year"
df_jmp_clean = df_jmp_clean.reset_index()
df_jmp_clean.insert(0, "country", "Kenya")
df_jmp_clean.insert(1, "country_code", "KEN")
df_jmp_clean.insert(2, "source", "JMP")

print(f"\nFinal shape: {df_jmp_clean.shape}")
df_jmp_clean.head()


Final shape: (31, 25)


,country,country_code,source,year,population_thousands_total,pct_improved_water_total,pct_at_least_basic_water_total,pct_limited_water_total,pct_unimproved_water_total,pct_surface_water_total,...,pct_unimproved_water_rural,pct_surface_water_rural,pct_piped_water_rural,population_thousands_urban,pct_improved_water_urban,pct_at_least_basic_water_urban,pct_limited_water_urban,pct_unimproved_water_urban,pct_surface_water_urban,pct_piped_water_urban
0,Kenya,KEN,JMP,2000,30642.890,52.1,45.4,6.7,17.4,30.5,...,20.0,37.3,18.5,6095.483398,89.9,85.4,4.5,6.7,3.4,83.2
1,Kenya,KEN,JMP,2001,31619.170,53.1,46.2,6.8,16.9,30.0,...,19.5,36.8,18.6,6399.403809,90.0,85.5,4.4,6.5,3.5,82.2
2,Kenya,KEN,JMP,2002,32629.808,54.0,47.1,6.9,16.4,29.5,...,19.0,36.3,18.8,6718.803711,90.1,85.7,4.4,6.4,3.5,81.3
3,Kenya,KEN,JMP,2003,33652.223,55.0,48.0,7.1,16.0,29.0,...,18.6,35.7,19.0,7049.467773,90.2,85.8,4.4,6.2,3.6,80.3
4,Kenya,KEN,JMP,2004,34713.455,56.0,48.8,7.2,15.5,28.5,...,18.1,35.2,19.1,7397.437500,90.3,86.0,4.3,6.0,3.7,79.3


In [43]:
# The four service levels should sum to ~100% for each year
check = df_jmp_clean[[
    "year",
    "pct_at_least_basic_water_total",
    "pct_limited_water_total",
    "pct_unimproved_water_total",
    "pct_surface_water_total",
]].copy()

check["sum"] = (
    check["pct_at_least_basic_water_total"].fillna(0) +
    check["pct_limited_water_total"].fillna(0) +
    check["pct_unimproved_water_total"].fillna(0) +
    check["pct_surface_water_total"].fillna(0)
)

print(check[["year", "sum"]].to_string(index=False))

 year   sum
 2000 100.0
 2001  99.9
 2002  99.9
 2003 100.1
 2004 100.0
 2005 100.1
 2006 100.0
 2007 100.0
 2008 100.1
 2009 100.0
 2010  99.9
 2011  99.9
 2012  99.9
 2013 100.0
 2014 100.0
 2015 100.1
 2016 100.0
 2017 100.0
 2018 100.1
 2019 100.0
 2020 100.0
 2021 100.0
 2022 100.0
 2023 100.0
 2024 100.0
 2025 100.0
 2026 100.1
 2027 100.0
 2028 100.0
 2029 100.0
 2030  99.9


In [44]:
from pathlib import Path

PROCESSED = Path("../data/processed")
output_path = PROCESSED / "jmp_kenya_clean.csv"
df_jmp_clean.to_csv(output_path, index=False)

print(f"Saved {len(df_jmp_clean)} rows → {output_path}")

Saved 31 rows → ..\data\processed\jmp_kenya_clean.csv


In [45]:
import datetime

CURRENT_YEAR = datetime.date.today().year

df_jmp_clean["is_estimated"] = df_jmp_clean["year"] > CURRENT_YEAR

print("Observed years:", df_jmp_clean[~df_jmp_clean["is_estimated"]]["year"].tolist())
print("Projected years:", df_jmp_clean[df_jmp_clean["is_estimated"]]["year"].tolist())

Observed years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Projected years: [2027, 2028, 2029, 2030]


In [46]:
# Define which columns are indicators vs metadata
JMP_META = ["country", "country_code", "source", "year", "is_estimated"]

JMP_VALUE_COLS = [c for c in df_jmp_clean.columns if c not in JMP_META]

df_jmp_long = df_jmp_clean.melt(
    id_vars=JMP_META,
    value_vars=JMP_VALUE_COLS,
    var_name="indicator_code",
    value_name="value"
)

# Drop NaNs
df_jmp_long = df_jmp_long.dropna(subset=["value"]).reset_index(drop=True)

print(f"JMP long shape: {df_jmp_long.shape}")
df_jmp_long.head(8)

JMP long shape: (651, 7)


,country,country_code,source,year,is_estimated,indicator_code,value
0,Kenya,KEN,JMP,2000,False,population_thousands_total,30642.890
1,Kenya,KEN,JMP,2001,False,population_thousands_total,31619.170
2,Kenya,KEN,JMP,2002,False,population_thousands_total,32629.808
3,Kenya,KEN,JMP,2003,False,population_thousands_total,33652.223
4,Kenya,KEN,JMP,2004,False,population_thousands_total,34713.455
5,Kenya,KEN,JMP,2005,False,population_thousands_total,35796.484
6,Kenya,KEN,JMP,2006,False,population_thousands_total,36904.008
7,Kenya,KEN,JMP,2007,False,population_thousands_total,38036.349


In [47]:
from pathlib import Path
import pandas as pd

PROCESSED = Path("../data/processed")

# Load the World Bank data you saved earlier
df_wb = pd.read_csv(PROCESSED / "wb_kenya_long.csv")

# Add the columns JMP has that WB doesn't
df_wb["source"]       = "World Bank"
df_wb["is_estimated"] = False
df_wb["country"]      = "Kenya"
df_wb["country_code"] = "KEN"

# Align column names
df_wb = df_wb.rename(columns={"Indicator Code": "indicator_code"}) \
             if "Indicator Code" in df_wb.columns \
             else df_wb

# Keep only the columns we need
WB_KEEP = ["country", "country_code", "source", "year",
           "is_estimated", "indicator_code", "value"]

df_wb_aligned = df_wb[[c for c in WB_KEEP if c in df_wb.columns]].copy()

print(f"World Bank aligned shape: {df_wb_aligned.shape}")
df_wb_aligned.head()

World Bank aligned shape: (49, 7)


,country,country_code,source,year,is_estimated,indicator_code,value
0,Kenya,KEN,World Bank,2000,False,ER.H2O.FWST.ZS,12.860042
1,Kenya,KEN,World Bank,2000,False,SH.STA.BASS.ZS,23.647380
2,Kenya,KEN,World Bank,2001,False,ER.H2O.FWST.ZS,14.948739
3,Kenya,KEN,World Bank,2001,False,SH.STA.BASS.ZS,24.316460
4,Kenya,KEN,World Bank,2002,False,ER.H2O.FWST.ZS,17.037436


In [48]:
df_master = pd.concat([df_jmp_long, df_wb_aligned], ignore_index=True)

# Sort for readability
df_master = df_master.sort_values(
    ["source", "indicator_code", "year"]
).reset_index(drop=True)

print(f"Master dataset shape: {df_master.shape}")
print(f"\nSources: {df_master['source'].value_counts().to_dict()}")
print(f"\nIndicators: {df_master['indicator_code'].nunique()} unique")
print(f"\nYear range: {df_master['year'].min()} to {df_master['year'].max()}")

Master dataset shape: (700, 7)

Sources: {'JMP': 651, 'World Bank': 49}

Indicators: 24 unique

Year range: 2000 to 2030


In [49]:
print("=== Missing values ===")
print(df_master.isnull().sum())

print("\n=== Rows per indicator ===")
print(
    df_master.groupby(["source", "indicator_code"])
    .agg(rows=("value", "count"),
         min_year=("year", "min"),
         max_year=("year", "max"))
    .to_string()
)

=== Missing values ===
country           0
country_code      0
source            0
year              0
is_estimated      0
indicator_code    0
value             0
dtype: int64

=== Rows per indicator ===
                                           rows  min_year  max_year
source     indicator_code                                          
JMP        pct_at_least_basic_water_rural    31      2000      2030
           pct_at_least_basic_water_total    31      2000      2030
           pct_at_least_basic_water_urban    31      2000      2030
           pct_improved_water_rural          31      2000      2030
           pct_improved_water_total          31      2000      2030
           pct_improved_water_urban          31      2000      2030
           pct_limited_water_rural           31      2000      2030
           pct_limited_water_total           31      2000      2030
           pct_limited_water_urban           31      2000      2030
           pct_piped_water_rural             31 

In [50]:
output_path = PROCESSED / "master_kenya_water.csv"
df_master.to_csv(output_path, index=False)

print(f"Saved {len(df_master)} rows → {output_path}")

Saved 700 rows → ..\data\processed\master_kenya_water.csv


## Data summary

- **Source 1: WHO/UNICEF JMP** — household estimates 2000–2030 (2025+ are projections).
  Covers drinking water service ladders (piped, basic, limited, unimproved, surface)
  split by Total, Rural, and Urban settings.

- **Source 2: World Bank** — 3 indicators with reliable Kenya data:
  - Basic sanitation coverage (2000–2024)
  - Water stress level (2000–2022)
  - WASH mortality rate (2019 only — limited utility)

- **Known gaps:** Safely managed drinking water (SH.H2O.SMDW.ZS) is absent from
  World Bank Kenya data. JMP's `pct_at_least_basic_water` series is the closest proxy.